# Notebook 3 – End-to-End ML Model on Tabular Data

In this notebook, we will:

- Take the cleaned **tabular dataset** from the previous notebook.
- Separate **features** (X) and **target** (y).
- Split the data into **train** and **test** sets.
- Train a simple **baseline ML model** (logistic regression).
- Evaluate it using accuracy (and briefly look at confusion matrix).

This gives us a complete, classical ML pipeline before we go deeper into TensorFlow and neural networks.


In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix


In [2]:
# ---------- OPTION A: Load cleaned CSV from previous notebook ----------
# Uncomment this if you saved "students_clean.csv" in Notebook 2.
# df = pd.read_csv("students_clean.csv")

# ---------- OPTION B: Re-create synthetic dataset (self-contained notebook) ----------
# If you prefer each notebook to stand alone, use this block instead.

np.random.seed(42)
n = 50  # slightly more rows now for a better split

data = {
    "student_id": np.arange(1, n + 1),
    "hours_studied": np.random.randint(0, 10, size=n).astype("float"),
    "previous_score": np.random.randint(30, 90, size=n).astype("float"),
    "attendance": np.random.randint(50, 100, size=n).astype("float"),
    "extra_classes": np.random.choice([0, 1], size=n).astype("float"),
    "final_score": np.random.randint(30, 95, size=n).astype("float"),
}

df = pd.DataFrame(data)

# Introduce some missing values
for col in ["hours_studied", "previous_score", "attendance"]:
    idx = np.random.choice(df.index, size=4, replace=False)
    df.loc[idx, col] = np.nan

# Simple mean imputation for numeric columns (same idea as Notebook 2)
num_cols = ["hours_studied", "previous_score", "attendance"]
for col in num_cols:
    df[col] = df[col].fillna(df[col].mean())

# Create a binary target: passed if final_score >= 50
df["passed"] = (df["final_score"] >= 50).astype(int)

df.head()


,student_id,hours_studied,previous_score,attendance,extra_classes,final_score,passed
0,1,6.0,76.0,56.000000,1.0,30.0,0
1,2,3.0,64.0,58.000000,1.0,56.0,1
2,3,7.0,43.0,73.000000,0.0,91.0,1
3,4,4.0,46.0,50.000000,0.0,32.0,0
4,5,6.0,65.0,72.347826,0.0,56.0,1


## 1. Problem setup: what are we predicting?

We will treat this as a simple **binary classification** problem:

- Input features:
  - `hours_studied`
  - `previous_score`
  - `attendance`
  - `extra_classes`
- Target label:
  - `passed` (1 = passed, 0 = failed), derived from `final_score`.

Goal:  
> Can we predict whether a student will pass or fail based on these features?

This mimics many real-world ML problems:

- Predict if a customer will **churn** or not.
- Predict if a transaction is **fraudulent** or not.
- Predict if a patient has a certain **disease** or not.


In [3]:
feature_cols = ["hours_studied", "previous_score", "attendance", "extra_classes"]
target_col = "passed"

X = df[feature_cols].values.astype("float32")
y = df[target_col].values.astype("float32")

X.shape, y.shape


((50, 4), (50,))

## 2. Train–test split: why we need it

If we train and evaluate our model on the **same data**, it can:

- Memorize patterns (even noise) in that data.
- Look very good in evaluation, but perform poorly on new, unseen data.

To simulate “future” or “unseen” data, we split our dataset into:

- **Training set**: used to fit the model (learn parameters).
- **Test set**: held back and only used at the end to evaluate performance.

Typical splits:

- 70% train / 30% test
- 80% train / 20% test, etc.

We’ll use scikit-learn’s `train_test_split()` to do this.


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,      # 20% test, 80% train
    random_state=42,    # for reproducibility
    stratify=y          # try to keep class balance similar in train and test
)

X_train.shape, X_test.shape, y_train.shape, y_test.shape


((40, 4), (10, 4), (40,), (10,))

### Class balance check

We quickly checked the fraction of:

- `passed = 0` (failed)
- `passed = 1` (passed)

in both train and test sets.

This helps us ensure that:

- The model sees a **representative mix** of both classes during training.
- The test set reflects roughly the same distribution as the original data.

If one class is extremely rare (highly imbalanced), accuracy alone can be misleading.
We’ll see more metrics later in the day.


In [5]:
print("Train label distribution (passed):")
print(pd.Series(y_train).value_counts(normalize=True))

print("\nTest label distribution (passed):")
print(pd.Series(y_test).value_counts(normalize=True))


Train label distribution (passed):
1.0    0.625
0.0    0.375
Name: proportion, dtype: float64

Test label distribution (passed):
1.0    0.6
0.0    0.4
Name: proportion, dtype: float64


## 3. Our first baseline model

We just trained a **logistic regression** model:

- It is a standard baseline model for binary classification.
- It works with numeric features, and is relatively interpretable.

Key things we checked:

- **Training accuracy**: how well the model fits the data it was trained on.
- **Test accuracy**: how well it generalizes to unseen data.

A good sign:

- Test accuracy is not dramatically worse than training accuracy.
- If training accuracy is very high but test accuracy is low, that usually indicates **overfitting**.


In [6]:
# Create and train a simple logistic regression model
log_reg = LogisticRegression()

log_reg.fit(X_train, y_train)

# Predictions on train and test
y_train_pred = log_reg.predict(X_train)
y_test_pred = log_reg.predict(X_test)

# Probabilities (optional, for later)
y_test_proba = log_reg.predict_proba(X_test)[:, 1]

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"Training accuracy: {train_acc:.2f}")
print(f"Test accuracy:     {test_acc:.2f}")


Training accuracy: 0.72
Test accuracy:     0.50


### Confusion matrix (quick intuition)

A **confusion matrix** for binary classification summarizes predictions as:

- **True Positives (TP)**: predicted pass, actually passed.
- **True Negatives (TN)**: predicted fail, actually failed.
- **False Positives (FP)**: predicted pass, actually failed.
- **False Negatives (FN)**: predicted fail, actually passed.

From this 2×2 table we can derive:

- **Accuracy** = (TP + TN) / (all).
- Later, we’ll also talk about **precision** and **recall** for certain problems.

For now, just remember:  
> The confusion matrix shows not only “how many we got right” but also the **type of mistakes** the model makes.


In [9]:
cm = confusion_matrix(y_test, y_test_pred)

cm


array([[1, 3],
       [2, 4]])

## 4. Why baselines matter

Before trying complex models (deep neural networks, ensembles, etc.), it is good practice to:

1. Define a **simple baseline**:
   - Predict the majority class.
   - Or a very simple linear/logistic model.

2. Check if your fancy model actually beats the baseline.

If a complex model does **not** clearly outperform a simple baseline, it may be:

- Overfitting.
- Poorly tuned.
- Or the extra complexity is simply not needed.

In our case, logistic regression should generally beat the “everyone passes” baseline by capturing patterns in the features.


In [10]:
# Very dumb baseline: always predict the majority class in the training set
majority_class = int(pd.Series(y_train).value_counts().idxmax())

y_test_dummy = np.full_like(y_test, fill_value=majority_class)
dummy_acc = accuracy_score(y_test, y_test_dummy)

print(f"Majority-class baseline accuracy: {dummy_acc:.2f}")
print(f"Our logistic regression test accuracy: {test_acc:.2f}")


Majority-class baseline accuracy: 0.60
Our logistic regression test accuracy: 0.50


## 5. Connecting this to TensorFlow

What we did in this notebook:

- Chose **features** (`hours_studied`, `previous_score`, `attendance`, `extra_classes`).
- Defined a **target** (`passed`).
- Split data into **train** and **test**.
- Trained a simple classical ML model (logistic regression).
- Evaluated it using accuracy and a confusion matrix.

In the next notebooks, we will:

- Build a similar end-to-end pipeline using **TensorFlow/Keras**.
- Replace the logistic regression with a **neural network** (dense layers).
- Compare performance and discuss when deep learning makes sense for tabular data.

For TensorFlow, the steps are conceptually the same:

1. Prepare `X_train`, `y_train`, `X_test`, `y_test`.
2. Build a Keras model.
3. `model.compile(...)`.
4. `model.fit(...)`.
5. `model.evaluate(...)`.

You’ve now seen the **classic ML pipeline**; next, we’ll just **change the model box**.
